# Original Code (adapted for mac mps)

In [27]:
import csv
import os
import traceback
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import torch
import ttach as tta
from PIL import Image
from skimage.measure import label as sklabel
from skimage.measure import regionprops
from skimage.transform import resize
import segmentation_models_pytorch_4TorchLessThan120 as smp


# LOAD DATA CONFIGS 

PROJECT_ROOT = Path(
    "/Users/JanayeCheong/Documents/radiomics_segmentation_models/"
    "TNSCUI2020-Seg-Rank1st"
)

IMG_DIR = PROJECT_ROOT / "train_thyroidXL" / "raw_images"
MASK_DIR = PROJECT_ROOT / "train_thyroidXL" / "masks"

WEIGHT_C1 = (
    PROJECT_ROOT
    / "weigh_and_id"
    / "TNSCUI"
    / "fold1_stage1_trained_on_size_256.pkl"
)

WEIGHT_C2 = (
    PROJECT_ROOT
    / "weigh_and_id"
    / "TNSCUI"
    / "fold1_stage2_trained_on_size_512.pkl"
)

OUTPUT_DIR = PROJECT_ROOT / "inference_outputs_long_vs_trans_v1_07"
PREDICTION_DIR = OUTPUT_DIR / "predicted_masks"
OVERLAY_DIR = OUTPUT_DIR / "overlays"
METRICS_CSV = OUTPUT_DIR / "metrics.csv"

C1_SIZE = 256
C2_SIZE = 512

C1_TTA = True
C2_TTA = True
USE_C2 = True

ORIMG = False
C1_THRESHOLD = 0.5
C2_THRESHOLD = 0.5
C2_RESIZE_ORDER = 0

SAVE_OVERLAYS = True
CONTINUE_ON_ERROR = True

SUPPORTED_EXTENSIONS = {
    ".png",
    ".jpg",
    ".jpeg",
    ".bmp",
    ".tif",
    ".tiff",
}



In [16]:
SCREENED_TXT = Path(
    "TNSCUI2020-Seg-Rank1st/train_thyroidXL/screened.txt"
)

VIEW_COMPARISON_DIR = OUTPUT_DIR / "view_comparisons"

VIEW_COMPARISON_CSV = OUTPUT_DIR / "view_comparison_by_patient.csv"
VIEW_SUMMARY_CSV = OUTPUT_DIR / "view_summary.csv"

In [25]:
def load_view_annotations(annotation_path: Path) -> Dict[str, str]:
    """
    Read screened.txt annotations.

    Expected format:
    # 8 digits + underscore + 8 digits + underscore + 1 digit
    
    first 8 digits indicate the patient ID

        ,classification
        00003307_17EE0585_0,Transverse 
        00003723_CCF3845C_2,Longitudinal
        ...

    Returns
    -------
    {
        "00003307_17EE0585_0": "Transverse",
        "00003723_CCF3845C_2": "Longitudinal",
        ...
    }
    """
    if not annotation_path.exists():
        raise FileNotFoundError(
            f"View annotation file not found: {annotation_path}"
        )

    annotations: Dict[str, str] = {}

    with annotation_path.open(
        "r",
        newline="",
        encoding="utf-8-sig",
    ) as f:

        reader = csv.reader(f)

        for row in reader:

            # Skip completely blank rows
            if not row:
                continue

            # Some files may contain blank lines
            if len(row) < 2:
                continue

            image_id = row[0].strip()
            classification = row[1].strip()

            # Skip header:
            # ,classification
            if (
                not image_id
                and classification.lower() == "classification"
            ):
                continue

            if not image_id:
                continue

            if not classification:
                classification = "Unknown"

            annotations[image_id.lower()] = classification

    print(
        f"Loaded view annotations for "
        f"{len(annotations)} images."
    )

    return annotations

In [ ]:
import pandas as pd

annotations_df = pd.read_csv(ANNOTATIONS_TXT, header=0, names=["image_name", "label"])

annotations_df['label'].value_counts()

"""
label
Transverse      387
Longitudinal    379
Unknown         183
Name: count, dtype: int64
"""

label
Transverse      387
Longitudinal    379
Unknown         183
Name: count, dtype: int64

In [17]:
annotations_df

,image_name,label
0,00003307_17EE0585_0,Transverse
1,00003723_CCF3845C_2,Longitudinal
2,00001962_034B467C_1,Transverse
3,00002300_E60B804B_0,Transverse
4,00003387_76B10567_2,Transverse
...,...,...
944,00000578_AAD6BA89_1,Unknown
945,00001334_CEB80814_1,Transverse
946,00003110_ABBD7CBC_0,Unknown
947,00001342_FFEB8295_0,Unknown


In [24]:
from collections import defaultdict, Counter
from pathlib import Path

IMG_DIR = PROJECT_ROOT / "train_thyroidXL" / "raw_images"

# Find images recursively and allow common extensions
image_files = [
    p for p in IMG_DIR.rglob("*")
    if p.is_file()
    and p.suffix.lower() in {".png", ".jpg", ".jpeg"}
]

print(f"Total image files found: {len(image_files)}")

# ------------------------------------------------------------
# Group by FIRST 8 CHARACTERS of filename stem
# ------------------------------------------------------------

patients = defaultdict(list)

for image_path in image_files:
    stem = image_path.stem.strip()

    patient_id = stem[:8]

    patients[patient_id].append(image_path)


# ------------------------------------------------------------
# Show all patients with MORE THAN ONE image
# ------------------------------------------------------------

repeated_patients = {
    patient_id: paths
    for patient_id, paths in patients.items()
    if len(paths) > 1
}

print(f"Unique patients: {len(patients)}")
print(
    f"Patients with >1 image: "
    f"{len(repeated_patients)}"
)

print("\n" + "=" * 70)
print("PATIENTS WITH MULTIPLE IMAGES")
print("=" * 70)

for patient_id, paths in sorted(
    repeated_patients.items(),
    key=lambda x: len(x[1]),
    reverse=True,
):

    print(
        f"\n{patient_id}: "
        f"{len(paths)} images"
    )

    for path in sorted(paths):
        print(f"    {path.name}")

Total image files found: 9541
Unique patients: 3354
Patients with >1 image: 3180

PATIENTS WITH MULTIPLE IMAGES

00004012: 10 images
    00004012_08B25F9C_9.png
    00004012_54792304_4.png
    00004012_60F3C0C2_1.png
    00004012_6E510AB4_8.png
    00004012_91D440E8_3.png
    00004012_AD725ABC_0.png
    00004012_BD239B56_7.png
    00004012_E1AFF0ED_2.png
    00004012_EBF1AAD9_5.png
    00004012_EC553CF6_6.png

00003730: 8 images
    00003730_13600386_2.png
    00003730_2D63A125_0.png
    00003730_429CBA29_5.png
    00003730_489C1FC9_3.png
    00003730_5A34EAFF_1.png
    00003730_5DA17244_3.png
    00003730_AD047E8B_4.png
    00003730_E247AD47_2.png

00003950: 8 images
    00003950_1B7C16DE_2.png
    00003950_2B4C084A_5.png
    00003950_2BE0C739_0.png
    00003950_330114F4_4.png
    00003950_50985A09_3.png
    00003950_953153A7_7.png
    00003950_A752ADE8_1.png
    00003950_CB339AA2_6.png

00004065: 8 images
    00004065_1B7A9C45_8.png
    00004065_77CA138D_7.png
    00004065_78063259_1

In [26]:
from collections import defaultdict, Counter
from pathlib import Path
import csv

SCREENED_TXT = Path(
    "TNSCUI2020-Seg-Rank1st/train_thyroidXL/screened.txt"
)

# ------------------------------------------------------------
# Group screened images by first 8 characters = patient ID
# ------------------------------------------------------------

patients = defaultdict(list)

with SCREENED_TXT.open(
    "r",
    newline="",
    encoding="utf-8-sig",
) as f:

    reader = csv.reader(f)

    for row in reader:

        if not row or len(row) < 2:
            continue

        image_id = row[0].strip()
        view = row[1].strip()

        # Skip header: ,classification
        if not image_id:
            continue

        patient_id = image_id[:8]

        patients[patient_id].append(
            {
                "image_id": image_id,
                "view": view,
            }
        )


# ------------------------------------------------------------
# Keep only repeat patients
# ------------------------------------------------------------

repeat_patients = {
    patient_id: images
    for patient_id, images in patients.items()
    if len(images) > 1
}


print(f"Total unique patients in screened file: {len(patients)}")
print(
    f"Repeat patients in screened file: "
    f"{len(repeat_patients)}"
)


# ------------------------------------------------------------
# Print each repeat patient and their images/views
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("REPEAT PATIENTS")
print("=" * 70)

for patient_id, images in sorted(
    repeat_patients.items(),
    key=lambda x: len(x[1]),
    reverse=True,
):

    print(
        f"\n{patient_id}: "
        f"{len(images)} screened images"
    )

    for item in images:
        print(
            f"    {item['image_id']} "
            f"| {item['view']}"
        )

FileNotFoundError: [Errno 2] No such file or directory: 'TNSCUI2020-Seg-Rank1st/train_thyroidXL/screened.txt'

In [15]:
from collections import defaultdict
from PIL import Image, ImageDraw

In [14]:
def thyroidxl_preprocess(
    image_path: Path,
    outputsize: C1_SIZE,
    remove_black_edges: bool = True,
):
    """
    ThyroidXL-safe replacement for TNSCUI_preprocess. 
    (original function in utils:
        removes irrelevant edges based on a threshold system of averaged pixels row-wise; the default output_size is 256 x 256
        returns the cut image tensor, coordinates of the min and max x and y FROM ORIGINAL IMAGE respectively that the extracted image without irrelevant
        areas contains  

    Returns
    -------
    processed_tensor:
        Float tensor with shape [output_size, output_size].

    cut_shape:
        Shape of the retained image before resizing.

    original_shape:
        Original image shape: (height, width).

    location:
        Coordinates of the retained image in the original image:
        [row_start, row_end, col_start, col_end].
    """

    # Force the image into one grayscale channel.
    with Image.open(image_path) as image:
        image = image.convert("L")
        image_array = np.asarray(image, dtype=np.float32)

    original_shape = image_array.shape

    if image_array.ndim != 2:
        raise ValueError(
            f"Expected a 2-D grayscale image, got {image_array.shape} "
            f"for {image_path.name}"
        )

    if remove_black_edges:
        # Detect rows and columns that contain meaningful ultrasound content.
        #
        # A small threshold is used instead of requiring pixels to be exactly
        # zero because ultrasound borders may contain compression noise.
        foreground_threshold = 5.0

        valid_rows = np.where(
            np.mean(image_array, axis=1) > foreground_threshold
        )[0]

        valid_cols = np.where(
            np.mean(image_array, axis=0) > foreground_threshold
        )[0]

        if len(valid_rows) > 0 and len(valid_cols) > 0:
            row_start = int(valid_rows[0])
            row_end = int(valid_rows[-1]) + 1

            col_start = int(valid_cols[0])
            col_end = int(valid_cols[-1]) + 1
        else:
            # Fall back to the complete image if no foreground is found.
            row_start = 0
            row_end = original_shape[0]
            col_start = 0
            col_end = original_shape[1]

    else:
        row_start = 0
        row_end = original_shape[0]
        col_start = 0
        col_end = original_shape[1]

    cropped_image = image_array[
        row_start:row_end,
        col_start:col_end,
    ]

    if cropped_image.size == 0:
        raise ValueError(
            f"Black-edge removal produced an empty image for "
            f"{image_path.name}"
        )

    cut_shape = cropped_image.shape
    location = [row_start, row_end, col_start, col_end]

    # Resize to the stage-1 
    processed_array = resize(
        cropped_image,
        (outputsize, outputsize),
        order=3,
        preserve_range=True,
        anti_aliasing=True,
    ).astype(np.float32)

    # Match the common neural-network image range.
    if processed_array.max() > 1.0:
        processed_array /= 255.0

    processed_tensor = torch.from_numpy(processed_array)

    return processed_tensor, cut_shape, original_shape, location

In [ ]:
# Metrics 

def get_iou(prediction: np.ndarray, ground_truth: np.ndarray) -> float:
    """Calculate intersection over union for two binary arrays --> corresponds to ."""
    prediction = prediction.astype(bool)
    ground_truth = ground_truth.astype(bool)

    intersection = np.logical_and(prediction, ground_truth).sum()
    union = np.logical_or(prediction, ground_truth).sum()

    if union == 0:
        return 1.0

    return float(intersection / union)


def get_dsc(prediction: np.ndarray, ground_truth: np.ndarray) -> float:
    """Calculate Dice similarity coefficient for two binary arrays."""
    prediction = prediction.astype(bool)
    ground_truth = ground_truth.astype(bool)

    intersection = np.logical_and(prediction, ground_truth).sum()
    denominator = prediction.sum() + ground_truth.sum()

    if denominator == 0:
        return 1.0

    return float((2.0 * intersection) / denominator)


def largest_connected_component(binary_mask: np.ndarray) -> np.ndarray:
    """Retain only the largest foreground connected component."""
    binary_mask = binary_mask.astype(bool)

    if binary_mask.sum() == 0:
        return binary_mask.astype(np.float32)

    labeled_img, num_components = sklabel(
        binary_mask,
        connectivity=1,
        background=0,
        return_num=True,
    )

    if num_components == 1:
        return binary_mask.astype(np.float32)

    component_sizes = [
        np.sum(labeled_img == component_label)
        for component_label in range(1, num_components + 1)
    ]

    largest_label = int(np.argmax(component_sizes)) + 1
    return (labeled_img == largest_label).astype(np.float32)


def calculate_stage2_roi(
    stage1_mask: np.ndarray,
    c1_size: int = 256,
) -> Tuple[int, int, int, int]:
    """
    Calculate the expanded square ROI used as input to Stage 2.

    Returns
    -------
    row_min, row_max, col_min, col_max
    """
    if stage1_mask.sum() == 0:
        min_row, min_col, max_row, max_col = 0, 0, c1_size, c1_size
    else:
        region = regionprops(stage1_mask.astype(np.uint8))[0]
        min_row, min_col, max_row, max_col = region.bbox

    row_center = (max_row + min_row) // 2
    col_center = (max_col + min_col) // 2
    max_length = max(max_row - min_row, max_col - min_col)

    large_roi_threshold = int((c1_size / 256) * 80)
    large_roi_margin = int((c1_size / 256) * 19)
    small_roi_margin = int((c1_size / 256) * 31)

    if max_length > large_roi_threshold:
        expansion = large_roi_margin + max_length // 2
    else:
        expansion = small_roi_margin + max_length // 2

    row_min = max(0, row_center - expansion)
    row_max = min(c1_size, row_center + expansion)
    col_min = max(0, col_center - expansion)
    col_max = min(c1_size, col_center + expansion)

    # IN case the crop is empty    
    if row_max <= row_min or col_max <= col_min:
        return 0, c1_size, 0, c1_size

    return row_min, row_max, col_min, col_max


# save files in order 

def discover_images(image_dir: Path) -> List[Path]:
    """Return every supported image file recursively, in stable order."""
    files = [
        path
        for path in image_dir.rglob("*")
        if path.is_file() and path.suffix.lower() in SUPPORTED_EXTENSIONS
    ]
    return sorted(files, key=lambda path: str(path).lower())


def build_mask_index(mask_dir: Path) -> Dict[str, Path]:
    """
    Index masks by filename stem, case-insensitively.

    The image and mask may have different filename extensions, but their stems
    must match
    """
    index: Dict[str, Path] = {}

    for path in discover_images(mask_dir):
        key = path.stem.lower()

        if key in index:
            raise ValueError(
                f"Duplicate mask stem '{path.stem}' found:\n"
                f"  {index[key]}\n"
                f"  {path}"
            )

        index[key] = path

    return index


def load_binary_mask(mask_path: Path, expected_shape: Tuple[int, int]) -> np.ndarray:
    """Read a mask as grayscale and convert it to a binary NumPy array."""
    mask = Image.open(mask_path).convert("L")
    mask_array = np.asarray(mask, dtype=np.float32)

    if mask_array.shape != expected_shape:
        raise ValueError(
            f"Ground-truth mask shape {mask_array.shape} does not match "
            f"original image shape {expected_shape} for {mask_path.name}."
        )

    return (mask_array > 0).astype(np.float32)


def save_binary_mask(mask: np.ndarray, output_path: Path) -> None:
    """Save a binary mask as an 8-bit PNG."""
    output_path.parent.mkdir(parents=True, exist_ok=True)
    mask_uint8 = (mask.astype(bool).astype(np.uint8) * 255)
    Image.fromarray(mask_uint8, mode="L").save(output_path)


def save_overlay(
    image_path: Path,
    prediction: np.ndarray,
    ground_truth: Optional[np.ndarray],
    output_path: Path,
) -> None:
    """
    Save an RGB overlay:
    - red: the prediction
    - green: the ground truth (from given mask)
    - yellow: overlap
    """
    original = Image.open(image_path).convert("L")
    base = np.asarray(original, dtype=np.float32)

    if base.max() > base.min():
        base = (base - base.min()) / (base.max() - base.min())
    else:
        base = np.zeros_like(base)

    rgb = np.stack([base, base, base], axis=-1)
    prediction_bool = prediction.astype(bool)

    rgb[prediction_bool, 0] = 1.0
    rgb[prediction_bool, 1] *= 0.35
    rgb[prediction_bool, 2] *= 0.35

    if ground_truth is not None:
        ground_truth_bool = ground_truth.astype(bool)
        rgb[ground_truth_bool, 1] = 1.0
        rgb[ground_truth_bool, 0] *= 0.35
        rgb[ground_truth_bool, 2] *= 0.35

        overlap = np.logical_and(prediction_bool, ground_truth_bool)
        rgb[overlap, 0] = 1.0
        rgb[overlap, 1] = 1.0
        rgb[overlap, 2] = 0.0

    output_path.parent.mkdir(parents=True, exist_ok=True)
    Image.fromarray((np.clip(rgb, 0, 1) * 255).astype(np.uint8), mode="RGB").save(
        output_path
    )


# ============================================================================
# Model loading and inference
# ============================================================================

def choose_device() -> torch.device:
    """Prefer Apple MPS, then CUDA, then CPU."""
    if torch.backends.mps.is_available():
        return torch.device("mps")

    if torch.cuda.is_available():
        return torch.device("cuda")

    return torch.device("cpu")


def extract_state_dict(checkpoint):
    """
    Support either a direct state_dict or a checkpoint dictionary containing
    a state_dict/model_state_dict field.
    """
    if not isinstance(checkpoint, dict):
        return checkpoint

    if "state_dict" in checkpoint:
        return checkpoint["state_dict"]

    if "model_state_dict" in checkpoint:
        return checkpoint["model_state_dict"]

    return checkpoint


def load_model(
    weight_path: Path,
    device: torch.device,
    transforms,
    use_tta: bool,
) -> torch.nn.Module:
    """Create a DeepLabV3+ model and load pretrained weights."""
    if not weight_path.exists():
        raise FileNotFoundError(f"Weight file not found: {weight_path}")

    model = smp.DeepLabV3Plus(
        encoder_name="efficientnet-b6",
        encoder_weights=None,
        in_channels=1,
        classes=1,
    )

    # CPU deserialization is typically the safest for old checkpoints.
    checkpoint = torch.load(weight_path, map_location="cpu")
    state_dict = extract_state_dict(checkpoint)

    try:
        model.load_state_dict(state_dict, strict=True)
    except RuntimeError as exc:
        print(
            f"Strict loading failed for {weight_path.name}; "
            "retrying with strict=False."
        )
        print(exc)
        incompatible = model.load_state_dict(state_dict, strict=False)
        print("Missing keys:", incompatible.missing_keys)
        print("Unexpected keys:", incompatible.unexpected_keys)

    model = model.to(device)
    model.eval()

    if use_tta:
        model = tta.SegmentationTTAWrapper(
            model,
            transforms,
            merge_mode="mean",
        )
        model.eval()

    return model


def run_single_image(
    image_path: Path,
    model_cascade1: torch.nn.Module,
    model_cascade2: torch.nn.Module,
    device: torch.device,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Run both cascades on one image.

    Returns
    -------
    final_mask:
        Binary mask restored to the original image dimensions.
    stage1_mask:
        Binary 256x256 Stage-1 largest-component mask.
    """
    processed_img, cut_shape, original_shape, location = thyroidxl_preprocess(
    image_path,
    outputsize=C1_SIZE,
    remove_black_edges=not ORIMG,
    )
    # Convert to a tensor if preprocessing returned a NumPy array.
    if processed_img.ndim != 2:
        raise ValueError(
            f"Expected processed image shape [H, W], got "
            f"{tuple(processed_img.shape)} for {image_path.name}"
        )

    image_tensor = processed_img.unsqueeze(0).unsqueeze(0)
    image_tensor = image_tensor.to(
        device=device,
        dtype=torch.float32,
    )

    assert image_tensor.shape == (1, 1, C1_SIZE, C1_SIZE), (
        f"Incorrect Stage-1 input shape: {tuple(image_tensor.shape)}"
    )

    image_array_256 = processed_img.cpu().numpy()

    with torch.inference_mode():
        stage1_logits = model_cascade1(image_tensor)
        stage1_probability = torch.sigmoid(stage1_logits)

    stage1_mask = (
        stage1_probability.squeeze().detach().cpu().numpy() > C1_THRESHOLD
    ).astype(np.float32)

    stage1_mask = largest_connected_component(stage1_mask)

    working_mask_256 = stage1_mask.copy()

    if USE_C2:
        row_min, row_max, col_min, col_max = calculate_stage2_roi(
            stage1_mask,
            C1_SIZE,
        )

        roi = image_array_256[row_min:row_max, col_min:col_max]
        roi_original_shape = roi.shape

        if roi.size == 0:
            raise RuntimeError(
                f"Stage-2 ROI is empty for {image_path.name}: "
                f"{(row_min, row_max, col_min, col_max)}"
            )

        roi_512 = resize(
            roi,
            (C2_SIZE, C2_SIZE),
            order=3,
            preserve_range=True,
            anti_aliasing=True,
        ).astype(np.float32)

        roi_tensor = torch.from_numpy(roi_512).unsqueeze(0).unsqueeze(0)
        roi_tensor = roi_tensor.to(device=device, dtype=torch.float32)

        with torch.inference_mode():
            stage2_logits = model_cascade2(roi_tensor)
            stage2_probability = torch.sigmoid(stage2_logits)

        stage2_mask_512 = (
            stage2_probability.squeeze().detach().cpu().numpy() > C2_THRESHOLD
        ).astype(np.float32)

        stage2_mask_roi = resize(
            stage2_mask_512,
            roi_original_shape,
            order=C2_RESIZE_ORDER,
            preserve_range=True,
            anti_aliasing=False,
        )

        stage2_mask_roi = (stage2_mask_roi > 0.5).astype(np.float32)

        # Use a blank canvas so the final 256x256 prediction contains only
        # the refined Stage-2 output, rather than leftover Stage-1 pixels.
        working_mask_256 = np.zeros((C1_SIZE, C1_SIZE), dtype=np.float32)
        working_mask_256[row_min:row_max, col_min:col_max] = stage2_mask_roi

    # Reverse the resize performed by TNSCUI_preprocess.
    restored_cut_mask = resize(
        working_mask_256,
        tuple(int(value) for value in cut_shape),
        order=0,
        preserve_range=True,
        anti_aliasing=False,
    )

    restored_cut_mask = (restored_cut_mask > 0.5).astype(np.float32)

    original_shape = tuple(int(value) for value in original_shape)
    final_mask = np.zeros(original_shape, dtype=np.float32)

    row_start, row_end, col_start, col_end = [int(value) for value in location]
    target_shape = (row_end - row_start, col_end - col_start)

    if restored_cut_mask.shape != target_shape:
        restored_cut_mask = resize(
            restored_cut_mask,
            target_shape,
            order=0,
            preserve_range=True,
            anti_aliasing=False,
        )
        restored_cut_mask = (restored_cut_mask > 0.5).astype(np.float32)

    final_mask[row_start:row_end, col_start:col_end] = restored_cut_mask
    final_mask = (final_mask > 0.5).astype(np.float32)

    return final_mask, stage1_mask


# ============================================================================
# Main dataset loop
# ============================================================================

def main() -> None:
    for required_path in (IMG_DIR, MASK_DIR):
        if not required_path.exists():
            raise FileNotFoundError(f"Directory not found: {required_path}")

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    PREDICTION_DIR.mkdir(parents=True, exist_ok=True)

    if SAVE_OVERLAYS:
        OVERLAY_DIR.mkdir(parents=True, exist_ok=True)

    device = choose_device()
    print(f"Using device: {device}")
    print(f"Image directory: {IMG_DIR}")
    print(f"Mask directory: {MASK_DIR}")

    image_files = discover_images(IMG_DIR)
    mask_index = build_mask_index(MASK_DIR)

    if not image_files:
        raise RuntimeError(f"No supported image files found in {IMG_DIR}")

    print(f"Found {len(image_files)} input images.")
    print(f"Found {len(mask_index)} masks.")

    missing_masks = [
        image_path.name
        for image_path in image_files
        if image_path.stem.lower() not in mask_index
    ]

    if missing_masks:
        preview = "\n".join(f"  - {name}" for name in missing_masks[:20])
        raise FileNotFoundError(
            f"{len(missing_masks)} images do not have matching masks by stem.\n"
            f"{preview}"
        )

    tta_transforms = tta.Compose(
        [
            tta.VerticalFlip(),
            tta.HorizontalFlip(),
            tta.Rotate90(angles=[0, 180]),
        ]
    )

    print("Loading Stage-1 model...")
    model_cascade1 = load_model(
        WEIGHT_C1,
        device,
        tta_transforms,
        C1_TTA,
    )

    print("Loading Stage-2 model...")
    model_cascade2 = load_model(
        WEIGHT_C2,
        device,
        tta_transforms,
        C2_TTA,
    )

    results: List[Dict] = []
    failed_count = 0

    for index, image_path in enumerate(image_files, start=1):
        mask_path = mask_index[image_path.stem.lower()]

        print(
            f"\n[{index}/{len(image_files)}] {image_path.name} "
            f"(C1={C1_SIZE}, C2={C2_SIZE})"
        )

        try:
            original_image = Image.open(image_path)
            original_shape = (original_image.height, original_image.width)

            ground_truth = load_binary_mask(mask_path, original_shape)

            final_mask, stage1_mask = run_single_image(
                image_path,
                model_cascade1,
                model_cascade2,
                device,
            )

            if final_mask.shape != ground_truth.shape:
                raise ValueError(
                    f"Prediction shape {final_mask.shape} does not match "
                    f"ground-truth shape {ground_truth.shape}."
                )

            iou = get_iou(final_mask, ground_truth)
            dsc = get_dsc(final_mask, ground_truth)

            prediction_path = PREDICTION_DIR / f"{image_path.stem}.png"
            save_binary_mask(final_mask, prediction_path)

            if SAVE_OVERLAYS:
                overlay_path = OVERLAY_DIR / f"{image_path.stem}_overlay.png"
                save_overlay(
                    image_path,
                    final_mask,
                    ground_truth,
                    overlay_path,
                )
            else:
                overlay_path = None

            result = {
                "image_name": image_path.name,
                "image_path": str(image_path),
                "mask_path": str(mask_path),
                "prediction_path": str(prediction_path),
                "overlay_path": str(overlay_path) if overlay_path else "",
                "height": original_shape[0],
                "width": original_shape[1],
                "stage1_foreground_pixels": int(stage1_mask.sum()),
                "final_foreground_pixels": int(final_mask.sum()),
                "ground_truth_foreground_pixels": int(ground_truth.sum()),
                "iou": iou,
                "dsc": dsc,
                "iou_below_0_3": int(iou < 0.3),
                "status": "ok",
                "error": "",
            }

            results.append(result)

            running_ious = [
                row["iou"] for row in results if row["status"] == "ok"
            ]
            running_dscs = [
                row["dsc"] for row in results if row["status"] == "ok"
            ]

            print(f"IoU: {iou:.4f}")
            print(f"DSC: {dsc:.4f}")
            print(f"Running mean IoU: {np.mean(running_ious):.4f}")
            print(f"Running mean DSC: {np.mean(running_dscs):.4f}")

        except Exception as exc:
            failed_count += 1
            print(f"ERROR processing {image_path.name}: {exc}")
            traceback.print_exc()

            results.append(
                {
                    "image_name": image_path.name,
                    "image_path": str(image_path),
                    "mask_path": str(mask_path),
                    "prediction_path": "",
                    "overlay_path": "",
                    "height": "",
                    "width": "",
                    "stage1_foreground_pixels": "",
                    "final_foreground_pixels": "",
                    "ground_truth_foreground_pixels": "",
                    "iou": "",
                    "dsc": "",
                    "iou_below_0_3": "",
                    "status": "failed",
                    "error": str(exc),
                }
            )

            if not CONTINUE_ON_ERROR:
                break

    fieldnames = [
        "image_name",
        "image_path",
        "mask_path",
        "prediction_path",
        "overlay_path",
        "height",
        "width",
        "stage1_foreground_pixels",
        "final_foreground_pixels",
        "ground_truth_foreground_pixels",
        "iou",
        "dsc",
        "iou_below_0_3",
        "status",
        "error",
    ]

    with METRICS_CSV.open("w", newline="", encoding="utf-8") as csv_file:
        writer = csv.DictWriter(csv_file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(results)

    successful_results = [
        row for row in results if row["status"] == "ok"
    ]

    print("\n" + "=" * 72)
    print("DATASET INFERENCE COMPLETE")
    print("=" * 72)
    print(f"Images discovered: {len(image_files)}")
    print(f"Successfully processed: {len(successful_results)}")
    print(f"Failed: {failed_count}")

    if successful_results:
        ious = np.asarray(
            [row["iou"] for row in successful_results],
            dtype=np.float64,
        )
        dscs = np.asarray(
            [row["dsc"] for row in successful_results],
            dtype=np.float64,
        )

        print(f"Mean IoU: {ious.mean():.4f}")
        print(f"Median IoU: {np.median(ious):.4f}")
        print(f"Mean DSC: {dscs.mean():.4f}")
        print(f"Median DSC: {np.median(dscs):.4f}")
        print(f"IoU below 0.3: {int(np.sum(ious < 0.3))}")

    print(f"Metrics CSV: {METRICS_CSV}")
    print(f"Predicted masks: {PREDICTION_DIR}")

    if SAVE_OVERLAYS:
        print(f"Overlays: {OVERLAY_DIR}")


if __name__ == "__main__":
    main()


Using device: mps
Image directory: /Users/JanayeCheong/Documents/radiomics_segmentation_models/TNSCUI2020-Seg-Rank1st/train_thyroidXL/raw_images
Mask directory: /Users/JanayeCheong/Documents/radiomics_segmentation_models/TNSCUI2020-Seg-Rank1st/train_thyroidXL/masks
Found 9541 input images.
Found 9541 masks.
Loading Stage-1 model...
Loading Stage-2 model...

[1/9541] 00000058_2201CE11_0.png (C1=256, C2=512)
IoU: 0.8404
DSC: 0.9133
Running mean IoU: 0.8404
Running mean DSC: 0.9133

[2/9541] 00000058_A73CED93_1.png (C1=256, C2=512)
IoU: 0.7828
DSC: 0.8781
Running mean IoU: 0.8116
Running mean DSC: 0.8957

[3/9541] 00000127_1914A778_1.png (C1=256, C2=512)
IoU: 0.7176
DSC: 0.8356
Running mean IoU: 0.7803
Running mean DSC: 0.8757

[4/9541] 00000127_313017DC_2.png (C1=256, C2=512)
IoU: 0.7887
DSC: 0.8819
Running mean IoU: 0.7824
Running mean DSC: 0.8772

[5/9541] 00000127_513DDC30_0.png (C1=256, C2=512)
IoU: 0.8032
DSC: 0.8909
Running mean IoU: 0.7865
Running mean DSC: 0.8800

[6/9541] 000001

KeyboardInterrupt: 

In [ ]:
from pathlib import Path

repo = Path("/content/sgh-segmodel")

matches = list(repo.rglob("segmentation_models_pytorch_4TorchLessThan120*"))

for match in matches:
    print(match)